# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the "FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` as per the Croissant specification.

In [ ]:
# Show all available record sets and their @ids
print("Available record sets:")
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
for rs_id in record_sets:
    print(f"- {rs_id}")

# For demonstration, list fields and columns in the first record set
if record_sets:
    first_rs_id = record_sets[0]
    # Access record set details
    rs_details = next(
        (rs for rs in dataset.metadata.to_json().get('recordSet', []) if rs['@id'] == first_rs_id),
        None
    )
    if rs_details:
        print(f"\nFields (by @id) in '{first_rs_id}':")
        for field in rs_details.get('field', []):
            print(f"  - {field['@id']}  Name: {field.get('name', '--')}  Type: {field.get('dataType', '--')}")
        print(f"\nColumns (by @id):")
        for col in rs_details.get('column', []):
            print(f"  - {col['@id']}  Name: {col.get('name', '--')}")

# Show a preview of some records by their @id
if record_sets:
    print(f"\nPreview records for record set {first_rs_id}:")
    for idx, record in enumerate(dataset.records(record_set=first_rs_id)):
        print(record)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

Use the record set and field `@id`s from the overview above.

In [ ]:
dataframes = {}
print("\nLoading record sets into DataFrames:")
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"- Record set: {rs_id}  Shape: {dataframes[rs_id].shape}")

# Select the primary record set for analysis
main_rs_id = record_sets[0] if record_sets else None

# Display columns, their @id, and preview
if main_rs_id:
    print(f"\nColumns (@id) in DataFrame for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll use the GAD-7 score (Generalized Anxiety Disorder scale), PHQ-9 (depression score), and demographic grouping fields.

In [ ]:
# Example: Identify relevant field @ids for numeric analysis
# We'll scan column names to match typical field names for GAD-7 or PHQ-9
# Assume columns like 'gad7_score', 'phq9_score', 'age', and 'village' exist, but reference with @id as required

df = dataframes[main_rs_id]
print("\nFields for numeric analysis (scan for typical scoring field names):")
print([col for col in df.columns if 'score' in col or 'age' in col])

# Let's pick 'gad7_score' and 'age' (by @id, if available)
numeric_field_id = 'gad7_score' if 'gad7_score' in df.columns else df.columns[0]
group_field_id = 'village' if 'village' in df.columns else df.columns[0]

# Filtering out records with GAD-7 score > 10
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the scores
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by village (if available)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No relevant numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, distribution of GAD-7 scores, or mean by village.

In [ ]:
# Visualize GAD-7 score distribution if numeric field found
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we have grouped_df, show mean score per village
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata and record sets using `mlcroissant`.
- Explored available fields and referenced all entities by their `@id`.
- Extracted data for primary record set and performed EDA on GAD-7 scores by village.
- Visualized distribution and highlighted potential high anxiety scores in certain villages.

This notebook demonstrates the reproducible AI-ready exploration of an African mental health survey dataset using standardized Croissant schema and tools.